In [ ]:
DATASET_PATH = ""

from pathlib import Path, PurePosixPath
import shutil
import zipfile

if not DATASET_PATH.strip():
    raise ValueError("Điền đường dẫn Kaggle Dataset vào DATASET_PATH.")

dataset = Path(DATASET_PATH.strip())
if not dataset.exists():
    raise FileNotFoundError(f"Không tìm thấy Dataset: {dataset}")

required_files = (
    "autograder.py", "backend.py", "models.py", "train.py", "losses.py",
    "data/mnist.npz", "data/lang_id.npz",
)
WORK_DIR = Path("/kaggle/working/project5_q1_q6")
WORK_DIR.mkdir(parents=True, exist_ok=True)

def copy_project(dataset):
    if dataset.is_dir():
        folders = sorted({
            path.parent for path in dataset.rglob("autograder.py")
            if all((path.parent / name).is_file() for name in required_files)
        })
        if len(folders) > 1:
            raise ValueError("Có nhiều thư mục project. Hãy trỏ DATASET_PATH vào thư mục cần chạy.")
        if folders:
            for name in required_files:
                destination = WORK_DIR / name
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(folders[0] / name, destination)
            return str(folders[0])
        archives = sorted(path for path in dataset.rglob("*") if path.is_file() and path.suffix.lower() == ".zip")
    else:
        archives = [dataset]

    projects = []
    for archive_path in archives:
        if not zipfile.is_zipfile(archive_path):
            continue
        with zipfile.ZipFile(archive_path) as archive:
            entries = {
                PurePosixPath(item.filename): item.filename
                for item in archive.infolist() if not item.is_dir()
            }
            for path in entries:
                if path.name == "autograder.py":
                    root = path.parent
                    if all(root / name in entries for name in required_files):
                        projects.append((archive_path, root, entries))

    if not projects:
        raise FileNotFoundError("Không tìm thấy source Q1–Q6 đầy đủ cùng data/mnist.npz và data/lang_id.npz.")
    if len(projects) > 1:
        raise ValueError("Có nhiều project trong các ZIP. Hãy trỏ DATASET_PATH vào ZIP chỉ chứa project cần chạy.")

    archive_path, root, entries = projects[0]
    with zipfile.ZipFile(archive_path) as archive:
        for name in required_files:
            destination = WORK_DIR / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entries[root / name]) as source, destination.open("wb") as output:
                shutil.copyfileobj(source, output)
    return f"{archive_path} ({root})"

print("Source:", copy_project(dataset))

In [ ]:
import importlib.util
import subprocess
import sys

missing = [name for name in ("numpy", "matplotlib", "torch") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

In [ ]:
import os
import re
import time

REPORT_DIR = Path("/kaggle/working/project5_q1_q6_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)
summary = []
env = dict(os.environ, MPLBACKEND="Agg", PYTHONIOENCODING="utf-8")

for question in ("q1", "q2", "q3", "q4", "q5", "q6"):
    print(f"\n========== {question.upper()} ==========", flush=True)
    log_path = REPORT_DIR / f"{question}.log"
    started = time.perf_counter()
    grade = None
    metric = None

    with log_path.open("w", encoding="utf-8") as log, subprocess.Popen(
        [sys.executable, "-u", "autograder.py", "-q", question, "--no-graphics"],
        cwd=WORK_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace",
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            match = re.fullmatch(rf"### Question {question}: (\d+/\d+) ###", line.strip())
            if match:
                grade = match.group(1)
            if line.startswith(("Your final loss", "Your final test set accuracy")):
                metric = line.strip()
        return_code = process.wait()

    elapsed = time.perf_counter() - started
    status = f"{grade} điểm" if grade else "không có điểm autograder"
    if return_code:
        status += f" | lỗi chạy ({return_code})"
    if metric:
        status += f" | {metric}"
    summary.append(f"{question.upper()} | {status} | {elapsed:.1f}s")

report = "\n".join(summary)
(REPORT_DIR / "summary.txt").write_text(report + "\n", encoding="utf-8")
print(f"\n{report}\n\nLogs: {REPORT_DIR}")